

#1. Gọi thư viện

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import unicodedata
import re

#2. Đọc tập tin dữ liệu

In [ ]:
# Đường dẫn file gốc
folder = Path("/content/drive/MyDrive/Tân Cảng Hiệp Phước/XU_LI_BAN_DAU")
file_path = folder / "TỔNG HỢP ĐÓNG RÚT HÀNG (2021-2025)-THỰC TẬP.xlsx"
# Đọc tất cả các sheet
df= pd.read_excel(file_path)

#3. Xử lý dữ liệu


##3.1 Loại bỏ các bản ghi không cần thiết

In [ ]:
df = df.iloc[3:].reset_index(drop=True)

df.columns = df.iloc[0]
df = df.iloc[1:].reset_index(drop=True)

##3.2 Lựa chọn thuộc tính và chuẩn hóa tên cột

In [ ]:
df = df[['NĂM', 'THÁNG', 'NGÀY', 'HÀNG\nHÓA', "Cont 40'", "Cont 20'"]].copy()

In [ ]:
# đổi tên cột
# Chuẩn hóa lại tên cột
df = df.rename(columns={
    'NĂM': 'Năm',
    'THÁNG': 'Tháng',
    'NGÀY': 'Ngày',
    'HÀNG\nHÓA': 'Loại hàng',
    "Cont 40'": 'Cont 40',
    "Cont 20'": 'Cont 20'
})

In [ ]:
# chỉ lấy các dòng có dữ liệu
# Lấy từ đầu đến hết hàng 17154 (vị trí 17151)
df_subset = df.iloc[:17152]

#3.3 Kiểm tra kiểu dữ liệu

In [ ]:
# Kiểm tra kiểu dữ liệu của tất cả các cột
print(df_subset.dtypes)

In [ ]:
df_subset.info()

In [ ]:
#Kiểm tra NaN
# Đếm số lượng NaN của từng cột
print(df_subset.isna().sum())

In [ ]:
# Lọc các dòng có cả Cont 40 và Cont 20 đều là NaN
df_missing_both = df_subset[df_subset['Cont 40'].isna() & df_subset['Cont 20'].isna()]

# Hiển thị kết quả
print(df_missing_both)
# Cách dùng len()
print(len(df_missing_both))

# Hoặc dùng .shape[0]
print(df_missing_both.shape[0])

In [ ]:
# xóa 5 hàng  cont 40 và cont 20 đều rỗgn
# Xóa các hàng có cả hai cột Cont 40 và Cont 20 đều là NaN trong df_subset
df_subset = df_subset.dropna(subset=['Cont 40', 'Cont 20'], how='all').copy()

# Kiểm tra lại số lượng dòng sau khi xóa
print("Số dòng còn lại:", len(df_subset))

In [ ]:
df_subset.info()

In [ ]:
df_subset

In [ ]:
# in ra giá trị unique của các cột ngày tháng năm
# In giá trị unique của cột Năm
print("Các giá trị unique của cột 'Năm':")
print(df_subset['Năm'].unique())
print("-" * 30)

# In giá trị unique của cột Tháng
print("Các giá trị unique của cột 'Tháng':")
print(df_subset['Tháng'].unique())
print("-" * 30)

# In giá trị unique của cột Ngày
print("Các giá trị unique của cột 'Ngày':")
print(df_subset['Ngày'].unique())

In [ ]:
# In giá trị unique của cột Cont 40 (sắp xếp tăng dần, bỏ qua NaN để dễ nhìn)
print("Các giá trị unique của cột 'Cont 40':")
print((df_subset['Cont 40'].dropna().unique()))
print("-" * 30)

# In giá trị unique của cột Cont 20 (sắp xếp tăng dần, bỏ qua NaN)
print("Các giá trị unique của cột 'Cont 20':")
print((df_subset['Cont 20'].dropna().unique()))

In [ ]:
# Lọc các dòng mà cột Cont 40 hoặc Cont 20 có giá trị là khoảng trắng ' '
df_space = df_subset[(df_subset['Cont 40'] == ' ') | (df_subset['Cont 20'] == ' ')]

# Hiển thị kết quả
print(f"Số lượng dòng chứa khoảng trắng: {len(df_space)}")
print(df_space)

In [ ]:
# 1. Thay thế chuỗi khoảng trắng ' ' thành NaN (hoặc dùng regex để bắt cả khoảng trắng ẩn)
df_subset['Cont 40'] = df_subset['Cont 40'].replace(r'^\s*$', pd.NA, regex=True)
df_subset['Cont 20'] = df_subset['Cont 20'].replace(r'^\s*$', pd.NA, regex=True)

# 2. Ép kiểu toàn bộ cột về dạng số (tự động loại bỏ/bỏ qua dấu ngoặc kép của string số và chuyển thành số nguyên/float)
df_subset['Cont 40'] = pd.to_numeric(df_subset['Cont 40'], errors='coerce')
df_subset['Cont 20'] = pd.to_numeric(df_subset['Cont 20'], errors='coerce')

# Kiểm tra lại kết quả unique sau khi làm sạch
print("Unique của 'Cont 40' sau khi chuẩn hóa:")
print(sorted(df_subset['Cont 40'].dropna().unique()))
print("-" * 40)
print("Unique của 'Cont 20' sau khi chuẩn hóa:")
print(sorted(df_subset['Cont 20'].dropna().unique()))

In [ ]:
# Chuyển đổi cột Cont 40 và Cont 20 về số nguyên (hỗ trợ cả giá trị NaN)
df_subset['Cont 40'] = df_subset['Cont 40'].astype('Int64')
df_subset['Cont 20'] = df_subset['Cont 20'].astype('Int64')

# Kiểm tra lại kiểu dữ liệu của các cột
print(df_subset[['Cont 40', 'Cont 20']].dtypes)

##2.4 Chuẩn hóa tên cột Loại hàng

In [ ]:
# Bước 1: Chuẩn hóa khoảng trắng và chữ hoa
def clean_text(x):
    if pd.isna(x):
        return None

    x = str(x).strip()
    x = re.sub(r'\s+', ' ', x)
    return x.upper()

df_subset['Loại hàng chuẩn hóa'] = (
    df_subset['Loại hàng'].apply(clean_text)
)

# Bước 2: Chuẩn hóa theo từ khóa
def normalize_goods(x):
    if pd.isna(x):
        return None

    # Mụn dừa
    if any(k in x for k in ['MỤN DỪA', 'MỤN DƯA', 'MUN DỪA']):
        return 'MỤN DỪA'

    # Gạo
    if 'GẠO' in x:
        return 'GẠO'

    # Máy móc
    if any(k in x for k in ['MÁY', 'MAÁY', 'CÀY', 'CĂT']):
        return 'MÁY MÓC'

    # Phân bón
    if 'PHÂN' in x:
        return 'PHÂN BÓN'

    # Xơ dừa
    if 'XƠ DỪA' in x or 'XƠ DƯA' in x:
        return 'XƠ DỪA'

    # Các nhóm còn lại
    mapping = {
        'LANH': 'LẠNH',
        'MÀNG PHIM': 'MÀNG NHỰA',
        'BÔT CÁ': 'BỘT CÁ',
        'BÁCH HÓA TỔNG HỢP': 'BÁCH HÓA',
        'KHÔ MỰC': 'HÀNG KHÔ',
        'MỰC KHÔ': 'HÀNG KHÔ',
        'THIẾT BỊ Đ.TỬ': 'THIẾT BỊ',
        'ÔTÔ': 'Ô TÔ',
        'XE OTO': 'Ô TÔ',
        'ÔTÔ QUÂN SỰ': 'Ô TÔ',
    }

    return mapping.get(x, x)


df_subset['Loại hàng chuẩn hóa'] = (
    df_subset['Loại hàng chuẩn hóa'].apply(normalize_goods)
)

# Bước 3: Thống kê kết quả
result = (
    df_subset['Loại hàng chuẩn hóa']
    .value_counts(dropna=False)
    .reset_index()
)

result.columns = ['Loại hàng chuẩn hóa', 'Số lượng']

result['Tỷ lệ (%)'] = (
    result['Số lượng'] / result['Số lượng'].sum() * 100
).round(2)

display(result)

In [ ]:
# Chuyển đổi các cột Năm, Tháng, Ngày về số nguyên (hỗ trợ cả NaN nếu có)
df_subset['Năm'] = df_subset['Năm'].astype('Int64')
df_subset['Tháng'] = df_subset['Tháng'].astype('Int64')
df_subset['Ngày'] = df_subset['Ngày'].astype('Int64')

# Kiểm tra lại kiểu dữ liệu của các cột
print(df_subset[['Năm', 'Tháng', 'Ngày']].dtypes)

In [ ]:
# (Tùy chọn) Chuyển về kiểu số nguyên sau khi đã fill 0
df_subset['Cont 40'] = df_subset['Cont 40'].astype(int)
df_subset['Cont 20'] = df_subset['Cont 20'].astype(int)

# 3. Tạo cột TEU bằng tổng Cont 20 và Cont 40 mới
df_subset['TEU'] = df_subset['Cont 20'] + df_subset['Cont 40']

# Kiểm tra lại kết quả vài dòng đầu tiên
print(df_subset[['Cont 20', 'Cont 40', 'TEU']].head())


In [ ]:
# Giữ lại các cột: Năm, Tháng, Ngày, Loại hàng và TEU
df_subset = df_subset[['Năm', 'Tháng', 'Ngày', 'Loại hàng','Loại hàng chuẩn hóa', 'TEU']].copy()

# Kiểm tra lại các cột sau khi lọc
print("Các cột còn lại trong DataFrame:")
print(df_subset.columns)

# Xem trước 5 dòng đầu của DataFrame mới
print("\Xem trước dữ liệu:")
print(df_subset.head())

#3. Xuất tập tin dữ liệu

In [ ]:
# Đường dẫn thư mục lưu file
folder = Path("/content/drive/MyDrive/Tân Cảng Hiệp Phước/XU_LI_BAN_DAU")


# Xuất file Excel
df_subset.to_excel(
    folder / "SL 2021_2025 clean.xlsx",
    index=False
)

print("Đã xuất file thành công!")